# Quantum QUBO Community Detection

**Approach**:
1. Load preprocessed neuronal network from `neurons.graphml`
2. Apply classical community detection baselines (Louvain, greedy modularity)
3. Formulate and solve modularity maximization as a QUBO problem using D-Wave
4. Compare results: modularity scores, runtime, cluster quality metrics
5. Visualize and analyze community structure across methods

### Import Libraries

In [35]:
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
import time

import dimod
from dwave.system import LeapHybridSampler
from dwave.system import DWaveSampler, EmbeddingComposite
from dwave.embedding import embed_bqm
from dwave.system import FixedEmbeddingComposite
from minorminer import find_embedding

from dimod import ExactSolver

from dwave.samplers import SimulatedAnnealingSampler

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)

### Load Data

In [2]:
# Load preprocessed network
G = nx.read_graphml('neurons.graphml')

print(f"Graph loaded: {G.number_of_nodes()} neurons, {G.number_of_edges()} connections")
print(f"Graph type: {type(G)}")
print(f"Is directed: {G.is_directed()}")

# Display sample node and edge attributes
sample_node = list(G.nodes())[0]
sample_edge = list(G.edges(data=True))[0]
print(f"\nSample node: {sample_node}")
print(f"Sample edge: {sample_edge}")

#prune graph to 70 nodes
def prune_graph(G, target_size=70):
    if G.number_of_nodes() <= target_size:
        return G.copy()
    
    # Compute degree centrality and sort nodes by it
    degree_centrality = nx.degree_centrality(G)
    sorted_nodes = sorted(degree_centrality, key=degree_centrality.get, reverse=True)
    
    # Keep only the top 'target_size' nodes
    nodes_to_keep = sorted_nodes[:target_size]
    pruned_G = G.subgraph(nodes_to_keep).copy()
    
    return pruned_G
G_pruned = prune_graph(G, target_size=70)

Graph loaded: 171 neurons, 466 connections
Graph type: <class 'networkx.classes.digraph.DiGraph'>
Is directed: True

Sample node: 0
Sample edge: ('0', '6', {'weight': 0.2168977552480034})


In [3]:
# Convert directed graph to undirected
G_undirected = G_pruned.to_undirected()

# Remove any remaining self-loops
G_undirected.remove_edges_from(nx.selfloop_edges(G_undirected))

# Create adjacency matrix
A = nx.adjacency_matrix(G_undirected).toarray()
N = G_undirected.number_of_nodes()
M = G_undirected.number_of_edges()  # Total number of edges

print(f"\nUndirected graph: {N} nodes, {M} edges")
print(f"Adjacency matrix shape: {A.shape}")
print(f"Sparsity: {(A == 0).sum() / A.size * 100:.2f}% zeros")
print(f"\nDegree statistics:")
degrees = [G_undirected.degree(n) for n in G_undirected.nodes()]
print(f"  Min: {min(degrees)}, Max: {max(degrees)}, Mean: {np.mean(degrees):.2f}")


Undirected graph: 70 nodes, 214 edges
Adjacency matrix shape: (70, 70)
Sparsity: 91.27% zeros

Degree statistics:
  Min: 1, Max: 39, Mean: 6.11


### Classical Community Detection

In [13]:
def communities_to_partition(communities):
    """
    Convert a list of community sets to a partition array.
    Returns: array where partition[i] = community_id for node i
    """
    node_list = list(G_undirected.nodes())  # Use the current graph's node order
    node_to_idx = {node: idx for idx, node in enumerate(node_list)}
    partition = np.zeros(len(node_list), dtype=int)
    for community_id, community_set in enumerate(communities):
        for node in community_set:
            partition[node_to_idx[node]] = community_id
    return partition

def compute_modularity(partition, A):
    """
    Compute modularity Q for a given partition.
    Q = (1/2m) * sum_ij (A_ij - k_i*k_j/2m) * delta(c_i, c_j)
    where k_i is the degree of node i, and delta(c_i, c_j) = 1 if nodes i,j in same community
    """
    degrees = A.sum(axis=1)
    m = A.sum() / 2  # total edges (double counted in adjacency matrix)
    
    Q = 0.0
    for i in range(N):
        for j in range(N):
            if partition[i] == partition[j]:
                Q += A[i, j] - (degrees[i] * degrees[j]) / (2 * m)
    
    Q /= (2 * m)
    return Q

def compute_conductance(partition, A):
    """
    Compute conductance for a given partition.
    Conductance = cut(S, S') / min(vol(S), vol(S'))
    where cut(S, S') is the number of edges between S and its complement,
    and vol(S) is the sum of degrees in S.
    """
    conductances = []
    for community_id in np.unique(partition):
        S = np.where(partition == community_id)[0]
        S_complement = np.where(partition != community_id)[0]
        
        cut = A[S][:, S_complement].sum()
        vol_S = A[S].sum()
        vol_S_complement = A[S_complement].sum()
        
        if min(vol_S, vol_S_complement) > 0:
            conductance = cut / min(vol_S, vol_S_complement)
            conductances.append(conductance)
    #add penalty to communities of size 1
    singleton_count = 0;
    for community_id in np.unique(partition):
        S = np.where(partition == community_id)[0]
        if len(S) == 1:
            singleton_count += 1  # Max conductance for single-node communities
    print(singleton_count)
    return (np.mean(conductances)+(singleton_count/N)) if conductances else 0.0

In [14]:
# Method 1: Louvain Community Detection (greedy optimization)
start_time = time.time()
communities_louvain = list(nx.community.louvain_communities(G_undirected, seed=42))
louvain_runtime = time.time() - start_time

partition_louvain = communities_to_partition(communities_louvain)
modularity_louvain = compute_modularity(partition_louvain, A)
conductance_louvain = compute_conductance(partition_louvain, A)

print(f"Runtime: {louvain_runtime:.4f}s")
print(f"Number of communities: {len(communities_louvain)}")
print(f"Modularity: {modularity_louvain:.6f}")
print(f"Conductance: {conductance_louvain:.6f}")
print(f"Community sizes: {sorted([len(c) for c in communities_louvain], reverse=True)}")

0
Runtime: 0.0144s
Number of communities: 8
Modularity: 0.349010
Conductance: 0.212923
Community sizes: [33, 11, 7, 6, 4, 3, 3, 3]


In [15]:
# Method 2: Greedy Modularity Optimization
start_time = time.time()
communities_greedy = list(nx.community.greedy_modularity_communities(G_undirected))
greedy_runtime = time.time() - start_time

partition_greedy = communities_to_partition(communities_greedy)
modularity_greedy = compute_modularity(partition_greedy, A)
conductance_greedy = compute_conductance(partition_greedy, A)

print(f"Runtime: {greedy_runtime:.4f}s")
print(f"Number of communities: {len(communities_greedy)}")
print(f"Modularity: {modularity_greedy:.6f}")
print(f"Conductance: {conductance_greedy:.6f}")
print(f"Community sizes: {sorted([len(c) for c in communities_greedy], reverse=True)}")

0
Runtime: 0.1633s
Number of communities: 6
Modularity: 0.337181
Conductance: 0.203194
Community sizes: [24, 17, 12, 10, 4, 3]


### Quantum QUBO Formulation and Solution

In [16]:
def build_modularity_qubo(A, num_communities=None):
    """
    Build QUBO matrix for modularity maximization via binary community assignment.
    
    Variables: x_{i,c} for i in nodes, c in communities
    Constraint: Each node assigned to exactly one community (handled via penalty)
    
    For simplicity with small network, use one-hot encoding:
    - num_binary_vars = N * num_communities
    - x[i*num_communities + c] = 1 if node i in community c
    
    QUBO objective: maximize modularity Q
    Reformulated as: minimize -Q + lambda * (constraint penalties)
    """
    
    if num_communities is None:
        num_communities = max(2, int(np.sqrt(N)))  # Heuristic: sqrt(N) communities
    
    n_vars = N * num_communities
    
    # Initialize QUBO matrix
    Q = {}
    
    degrees = A.sum(axis=1)
    m = A.sum() / 2
    
    # Modularity term: maximize sum_ij (A_ij - k_i*k_j/2m) * x_{i,c} * x_{j,c}
    for c in range(num_communities):
        for i in range(N):
            for j in range(N):
                var_i = i * num_communities + c
                var_j = j * num_communities + c
                
                if i == j:
                    # Diagonal terms (linear)
                    coeff = -2 * (1 + (A[i, i] - degrees[i]**2 / (2*m))) / (2*m)
                    if var_i not in Q:
                        Q[var_i] = 0
                    Q[var_i] += coeff
                elif i < j:
                    # Off-diagonal (quadratic)
                    coeff = -2 * (A[i, j] - degrees[i] * degrees[j] / (2*m)) / (2*m)
                    if (var_i, var_j) not in Q:
                        Q[(var_i, var_j)] = 0
                    Q[(var_i, var_j)] += coeff
    
    # Constraint penalty: each node in exactly one community
    # Penalty: lambda * sum_i (sum_c x_{i,c} - 1)^2
    lambda_penalty = 10.0
    
    for i in range(N):
        # (sum_c x_{i,c} - 1)^2 = sum_c x_{i,c}^2 - 2*sum_c x_{i,c} + 1
        #                        = sum_c x_{i,c} - 2*sum_c x_{i,c} + 1  (since x^2 = x for binary)
        #                        = 1 - sum_c x_{i,c}
        for c in range(num_communities):
            var_i = i * num_communities + c
            if var_i not in Q:
                Q[var_i] = 0
            Q[var_i] += -2 * lambda_penalty
        
        # Cross terms in penalty
        for c1 in range(num_communities):
            for c2 in range(c1+1, num_communities):
                var_1 = i * num_communities + c1
                var_2 = i * num_communities + c2
                key = (var_1, var_2) if var_1 < var_2 else (var_2, var_1)
                if key not in Q:
                    Q[key] = 0
                Q[key] += 2 * lambda_penalty
    
    return Q, num_communities

In [17]:
# Estimate reasonable number of communities
num_communities_target = max(2, int(np.sqrt(N)))
print(f"Target number of communities: {num_communities_target}")
print(f"Total QUBO variables: {N * num_communities_target}")

# Build QUBO
print(f"\nBuilding QUBO matrix...")
Q_dict, k_communities = build_modularity_qubo(A, num_communities_target)
print(f"QUBO built with {len(Q_dict)} terms")

Target number of communities: 8
Total QUBO variables: 560

Building QUBO matrix...
QUBO built with 21840 terms


In [18]:
# Convert QUBO to BQM (Binary Quadratic Model) for D-Wave
Q_dict_clean = {}
for k, v in Q_dict.items():
    if isinstance(k, tuple):
        Q_dict_clean[k] = float(v)
    else:
        node_idx = int(k)
        Q_dict_clean[(node_idx, node_idx)] = float(v)
bqm = dimod.BinaryQuadraticModel.from_qubo(Q_dict_clean)
print(f"BQM created: {bqm.num_variables} variables, {len(bqm.quadratic)} interactions")

BQM created: 560 variables, 21280 interactions


In [19]:
# Method 3a: Exact Solver (for baseline/verification on small problem)
if N * k_communities <= 20:  # Only practical for very small problems
    start_time = time.time()
    exact_solver = ExactSolver()
    sampleset_exact = exact_solver.sample(bqm)
    exact_runtime = time.time() - start_time
    
    best_sample_exact = sampleset_exact.first.sample
    best_energy_exact = sampleset_exact.first.energy
    
    print(f"Runtime: {exact_runtime:.4f}s")
    print(f"Best energy (lower is better): {best_energy_exact:.6f}")
    print(f"Exact solver can evaluate all 2^{bqm.num_variables} states")
else:
    print(f"Problem too large ({N * k_communities} variables) for exact solver.")

Problem too large (560 variables) for exact solver.


In [20]:
# Method 3b: Simulated Annealing (D-Wave classical emulation); simplified A LOT because 100 reads takes too long
start_time = time.time()
sa_sampler = SimulatedAnnealingSampler()
sampleset_sa = sa_sampler.sample(bqm, num_reads=100, seed=42)
sa_runtime = time.time() - start_time

best_sample_sa = sampleset_sa.first.sample
best_energy_sa = sampleset_sa.first.energy

print(f"Runtime: {sa_runtime:.4f}s")
print(f"Best energy: {best_energy_sa:.6f}")
print(f"Number of reads: 100")
print(f"Top 5 solutions:")
#convert generator into list
top5 = list(sampleset_sa.data(['sample', 'energy', 'num_occurrences'], sorted_by='energy'))[:5]
for i, (sample, energy, num_occurrences) in enumerate(top5):
    print(f"  {i+1}. Energy: {energy:.6f}, Occurrences: {num_occurrences}")

Runtime: 3.7474s
Best energy: -1403.315358
Number of reads: 100
Top 5 solutions:
  1. Energy: -1403.315358, Occurrences: 1
  2. Energy: -1403.310052, Occurrences: 1
  3. Energy: -1403.307502, Occurrences: 1
  4. Energy: -1403.297714, Occurrences: 1
  5. Energy: -1403.296717, Occurrences: 1


In [21]:
def extract_communities_from_sample(sample, N, k_communities):
    """
    Extract community assignments from QUBO sample.
    For node i, find c where x_{i,c} = 1
    """
    partition = np.zeros(N, dtype=int)
    
    for i in range(N):
        # Find which community this node is assigned to
        for c in range(k_communities):
            var_idx = i * k_communities + c
            if var_idx in sample and sample[var_idx] == 1:
                partition[i] = c
                break
        else:
            # If no community assigned (constraint violation), assign to community 0
            partition[i] = 0
    
    return partition
count = 1

for sample, energy, num_occurrences in top5:
    partition_sa = extract_communities_from_sample(sample, N, k_communities)
    modularity_sa = compute_modularity(partition_sa, A)
    num_communities_found_sa = len(np.unique(partition_sa))
    
    print(f"\nPartition from sample with energy {energy:.6f}:")
    print(f"\nSample {count}:")
    print(f"  Number of communities found: {num_communities_found_sa}")
    print(f"  Modularity: {modularity_sa:.6f}")
    print(f"  Community sizes: {sorted(np.bincount(partition_sa), reverse=True)}")
    count += 1


Partition from sample with energy -1403.315358:

Sample 1:
  Number of communities found: 7
  Modularity: 0.098307
  Community sizes: [np.int64(23), np.int64(13), np.int64(13), np.int64(9), np.int64(8), np.int64(2), np.int64(2)]

Partition from sample with energy -1403.310052:

Sample 2:
  Number of communities found: 7
  Modularity: 0.076479
  Community sizes: [np.int64(20), np.int64(14), np.int64(13), np.int64(10), np.int64(9), np.int64(3), np.int64(1)]

Partition from sample with energy -1403.307502:

Sample 3:
  Number of communities found: 7
  Modularity: 0.056277
  Community sizes: [np.int64(18), np.int64(12), np.int64(12), np.int64(10), np.int64(9), np.int64(6), np.int64(3)]

Partition from sample with energy -1403.297714:

Sample 4:
  Number of communities found: 7
  Modularity: 0.085678
  Community sizes: [np.int64(19), np.int64(18), np.int64(15), np.int64(8), np.int64(7), np.int64(2), np.int64(1)]

Partition from sample with energy -1403.296717:

Sample 5:
  Number of commun

In [30]:
#before using d-wave hybrid solve,check feature availability
Dsampler = DWaveSampler()
print("Connected to sampler", Dsampler.solver.name)

Connected to sampler Advantage_system4.1


In [36]:
#greedy optimization
def greedy_refinement(A, partition):
    """
    Refines the partition using a greedy local search.
    Moves each node to the community that gives the highest modularity increase.
    """
    refined_partition = partition.copy()
    N = A.shape[0]
    unique_communities = np.unique(partition)
    improved = True
    
    current_modularity = compute_modularity(refined_partition, A)
    
    print(f"Starting refinement... Initial Modularity: {current_modularity:.6f}")
    
    while improved:
        improved = False
        nodes = np.random.permutation(N)
        
        for i in nodes:
            best_move = refined_partition[i]
            initial_comm = refined_partition[i]
            for target_comm in unique_communities:
                if target_comm == initial_comm:
                    continue
                refined_partition[i] = target_comm
                new_modularity = compute_modularity(refined_partition, A)
                if new_modularity > current_modularity:
                    current_modularity = new_modularity
                    best_move = target_comm
                    improved = True
            refined_partition[i] = best_move
            
    print(f"Refinement complete. Optimized Modularity: {current_modularity:.6f}")
    return refined_partition

In [38]:
# Method 3c: D-Wave LeapHybrid Solver (quantum+classical, requires Leap account)
start_time = time.time()
sampler = LeapHybridSampler()
sampleset_hybrid = sampler.sample(bqm, time_limit=10)

#target_edgelist = sampler.edgelist
#find_embedding(bqm.quadratic, target_edgelist)

# sampleset_hybrid = hybrid_sampler.sample(bqm, time_limit=1)
hybrid_runtime = time.time() - start_time

best_sample_hybrid = sampleset_hybrid.first.sample
best_energy_hybrid = sampleset_hybrid.first.energy
# use_hybrid = True

In [39]:
# Method 3c: print parameters

#refine w greedy optimization
partition_hybrid_raw = extract_communities_from_sample(best_sample_hybrid, N, k_communities)
print("Applying greedy refinement to hybrid solution...")
partition_hybrid_optimized = greedy_refinement(A, partition_hybrid_raw)
partition_hybrid = extract_communities_from_sample(best_sample_hybrid, N, k_communities)

num_communities_found_hybrid = len(np.unique(partition_hybrid))
conductance_hybrid = compute_conductance(partition_hybrid, A)
mod_raw = compute_modularity(partition_hybrid_raw, A)
mod_opt = compute_modularity(partition_hybrid_optimized, A)

print(f"Runtime: {hybrid_runtime:.4f}s")
print(f"Best energy: {best_energy_hybrid:.6f}")
print(f"Number of communities found: {num_communities_found_hybrid}")
print(f"Original Hybrid Modularity: {mod_raw:.6f}")
print(f"Optimized Hybrid Modularity: {mod_opt:.6f}")
print(f"Conductance: {conductance_hybrid:.6f}")
print(f"Community sizes: {sorted(np.bincount(partition_hybrid), reverse=True)}")

Applying greedy refinement to hybrid solution...
Starting refinement... Initial Modularity: 0.285263
Refinement complete. Optimized Modularity: 0.332837
1
Runtime: 2.2887s
Best energy: -1403.701648
Number of communities found: 6
Original Hybrid Modularity: 0.285263
Optimized Hybrid Modularity: 0.332837
Conductance: 0.550477
Community sizes: [np.int64(25), np.int64(15), np.int64(12), np.int64(12), np.int64(5), np.int64(1), np.int64(0)]


In [31]:
#3d. using the DWave QPU sampler (requires D-Wave account and access to quantum hardware)
sampler_3d = EmbeddingComposite(Dsampler)

print(f"Submitting to QPU: {Dsampler.properties['chip_id']}")

# submit the same QUBO to the QPU and measure runtime
start_time = time.time()
sampleset_qpu = sampler_3d.sample_qubo(Q_dict_clean, num_reads=1000, chain_strength=None)
qpu_runtime = time.time() - start_time

best_sample_qpu = sampleset_qpu.first.sample
partition_qpu = np.zeros(N, dtype=int)


Submitting to QPU: Advantage_system4.1


KeyboardInterrupt: embedding cancelled by keyboard interrupt

In [ ]:
#3d postprocessing
for i in range(N):
    for c in range(k_communities):
        if best_sample_qpu[i * k_communities + c] == 1:
            partition_qpu[i] = c
            break

modularity_qpu = compute_modularity(partition_qpu, A)
conductance_qpu = compute_conductance(partition_qpu, A)

print(f"QPU Runtime: {qpu_runtime:.4f}s")
print(f"QPU Modularity: {modularity_qpu:.6f}")
print(f"QPU Conductance: {conductance_qpu:.6f}")

### Comparison

In [ ]:
# Create comparison dataframe
comparison_data = {
    'Method': ['Louvain', 'Greedy Modularity', 'Simulated Annealing'],
    'Modularity': [modularity_louvain, modularity_greedy, modularity_sa],
    'Num Communities': [len(communities_louvain), len(communities_greedy), num_communities_found_sa],
    'Runtime (s)': [louvain_runtime, greedy_runtime, sa_runtime],
    'Type': ['Classical', 'Classical', 'Quantum-Inspired']
}

comparison_data['Method'].append('D-Wave Hybrid')
comparison_data['Modularity'].append(modularity_hybrid)
comparison_data['Num Communities'].append(num_communities_found_hybrid)
comparison_data['Runtime (s)'].append(hybrid_runtime)
comparison_data['Type'].append('Quantum')

df_comparison = pd.DataFrame(comparison_data)
print("\nCOMMUNITY DETECTION COMPARISON")
print("="*80)
print(df_comparison.to_string(index=False))
print("="*80)

# Highlight best modularity
best_idx = df_comparison['Modularity'].idxmax()
print(f"\nBest modularity: {df_comparison.loc[best_idx, 'Method']} with Q={df_comparison.loc[best_idx, 'Modularity']:.6f}")

In [ ]:
# Compute pairwise Agreement metrics between partitions
partitions = {
    'Louvain': partition_louvain,
    'Greedy': partition_greedy,
    'Simulated Annealing': partition_sa
}

partitions['D-Wave Hybrid'] = partition_hybrid

# Compute Adjusted Rand Index (ARI) between all pairs
methods = list(partitions.keys())
ari_matrix = np.zeros((len(methods), len(methods)))
nmi_matrix = np.zeros((len(methods), len(methods)))

for i, method1 in enumerate(methods):
    for j, method2 in enumerate(methods):
        ari_matrix[i, j] = adjusted_rand_score(partitions[method1], partitions[method2])
        nmi_matrix[i, j] = normalized_mutual_info_score(partitions[method1], partitions[method2])

df_ari = pd.DataFrame(ari_matrix, index=methods, columns=methods)
df_nmi = pd.DataFrame(nmi_matrix, index=methods, columns=methods)

print("\nADJUSTED RAND INDEX (ARI) - Agreement between partitions")
print("Range: [-1, 1], where 1 = perfect agreement, 0 = random agreement")
print("="*60)
print(df_ari.round(4))

print("\nNORMALIZED MUTUAL INFORMATION (NMI)")
print("Range: [0, 1], where 1 = perfect agreement")
print("="*60)
print(df_nmi.round(4))

In [ ]:
# Compute network quality metrics per method
def compute_conductance(partition, A):
    """
    Compute conductance for each community (measure of internal cohesion).
    Lower conductance = better separated communities
    """
    conductances = []
    
    for c in np.unique(partition):
        nodes_in_community = np.where(partition == c)[0]
        
        if len(nodes_in_community) <= 1:
            conductances.append(0)
            continue
        
        # Edges cut (between community and rest)
        edges_cut = 0
        for i in nodes_in_community:
            for j in range(N):
                if partition[j] != c and A[i, j] > 0:
                    edges_cut += A[i, j]
        
        # Min of internal degree sum
        internal_degree = A[np.ix_(nodes_in_community, nodes_in_community)].sum()
        total_degree = A[nodes_in_community].sum()
        
        denom = min(internal_degree, total_degree - internal_degree)
        
        conductance = edges_cut / denom if denom > 0 else 0
        conductances.append(conductance)
    
    return np.mean(conductances) if conductances else 0

quality_metrics = {}
for method, partition in partitions.items():
    num_comms = len(np.unique(partition))
    modularity = compute_modularity(partition, A)
    conductance = compute_conductance(partition, A)
    
    quality_metrics[method] = {
        'Modularity': modularity,
        'Num Communities': num_comms,
        'Avg Conductance': conductance
    }

df_quality = pd.DataFrame(quality_metrics).T
print("\nCOMMUNITY QUALITY METRICS")
print("="*60)
print(df_quality.round(6))

### Visualizations

In [ ]:
# Visualization 1: Comparison of Modularity Scores
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart: Modularity comparison
methods = df_comparison['Method'].values
modularities = df_comparison['Modularity'].values
colors = ['#1f77b4' if t == 'Classical' else '#ff7f0e' if t == 'Quantum-Inspired' else '#2ca02c' 
          for t in df_comparison['Type'].values]

axes[0].bar(methods, modularities, color=colors, alpha=0.7, edgecolor='black')
axes[0].set_ylabel('Modularity Q', fontsize=12)
axes[0].set_title('Community Detection: Modularity Comparison', fontsize=13, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45)
axes[0].grid(axis='y', alpha=0.3)

for i, (method, mod) in enumerate(zip(methods, modularities)):
    axes[0].text(i, mod + 0.005, f'{mod:.4f}', ha='center', va='bottom', fontsize=10)

# Bar chart: Runtime comparison (log scale)
runtimes = df_comparison['Runtime (s)'].values
axes[1].bar(methods, runtimes, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Runtime (seconds, log scale)', fontsize=12)
axes[1].set_title('Community Detection: Runtime Comparison', fontsize=13, fontweight='bold')
axes[1].set_yscale('log')
axes[1].tick_params(axis='x', rotation=45)
axes[1].grid(axis='y', alpha=0.3)

for i, (method, rt) in enumerate(zip(methods, runtimes)):
    axes[1].text(i, rt * 1.2, f'{rt:.4f}s', ha='center', va='bottom', fontsize=10)

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1f77b4', alpha=0.7, label='Classical'),
                   Patch(facecolor='#ff7f0e', alpha=0.7, label='Quantum-Inspired'),
                   Patch(facecolor='#2ca02c', alpha=0.7, label='Quantum')]
fig.legend(handles=legend_elements, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3, fontsize=11)

plt.tight_layout()
plt.savefig('community_detection_comparison.png', dpi=100, bbox_inches='tight')
plt.show()

print("Saved: community_detection_comparison.png")

In [ ]:
# Visualization 2: Heatmaps of Agreement Metrics
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ARI heatmap
sns.heatmap(df_ari, annot=True, fmt='.3f', cmap='RdYlGn', vmin=-1, vmax=1, ax=axes[0], cbar_kws={'label': 'ARI'})
axes[0].set_title('Adjusted Rand Index (ARI) - Partition Agreement', fontsize=13, fontweight='bold')

# NMI heatmap
sns.heatmap(df_nmi, annot=True, fmt='.3f', cmap='RdYlGn', vmin=0, vmax=1, ax=axes[1], cbar_kws={'label': 'NMI'})
axes[1].set_title('Normalized Mutual Information (NMI) - Partition Agreement', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('partition_agreement_metrics.png', dpi=100, bbox_inches='tight')
plt.show()

print("Saved: partition_agreement_metrics.png")

In [ ]:
# Visualization 3: Network Visualization with Community Colors (for best method)
best_method = df_comparison.loc[df_comparison['Modularity'].idxmax(), 'Method']
best_partition = partitions[best_method]

fig, axes = plt.subplots(2, 2, figsize=(16, 14))

for idx, (method, partition) in enumerate(list(partitions.items())[:4]):
    ax = axes[idx // 2, idx % 2]
    
    # Use spring layout for visualization
    pos = nx.spring_layout(G_undirected, seed=42, k=0.5, iterations=50)
    
    # Node colors based on community
    node_colors = [partition[i] for i in range(N)]
    num_unique_communities = len(np.unique(partition))
    
    # Draw network
    nx.draw_networkx_edges(G_undirected, pos, ax=ax, alpha=0.2, width=0.5)
    nodes = nx.draw_networkx_nodes(G_undirected, pos, ax=ax, 
                                   node_color=node_colors, 
                                   node_size=200, 
                                   cmap='tab20',
                                   vmin=0, vmax=max(node_colors))
    
    modularity_val = compute_modularity(partition, A)
    title = f'{method}\nModularity: {modularity_val:.4f}, Communities: {num_unique_communities}'
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.axis('off')

plt.tight_layout()
plt.savefig('community_networks_visualization.png', dpi=100, bbox_inches='tight')
plt.show()

print("Saved: community_networks_visualization.png")

In [ ]:
# Visualization 4: Community Size Distribution
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for idx, (method, partition) in enumerate(list(partitions.items())[:4]):
    ax = axes[idx // 2, idx % 2]
    
    community_sizes = np.bincount(partition)
    communities = np.arange(len(community_sizes))
    
    ax.bar(communities, community_sizes, color='steelblue', alpha=0.7, edgecolor='black')
    ax.set_xlabel('Community ID', fontsize=11)
    ax.set_ylabel('Number of Neurons', fontsize=11)
    ax.set_title(f'{method} - Community Sizes', fontsize=12, fontweight='bold')
    ax.grid(axis='y', alpha=0.3)
    
    # Add statistics
    stats_text = f'Mean: {np.mean(community_sizes):.1f}\nStd: {np.std(community_sizes):.1f}\nMin: {np.min(community_sizes)}\nMax: {np.max(community_sizes)}'
    ax.text(0.98, 0.97, stats_text, transform=ax.transAxes, 
            fontsize=9, verticalalignment='top', horizontalalignment='right',
            bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig('community_size_distributions.png', dpi=100, bbox_inches='tight')
plt.show()

print("Saved: community_size_distributions.png")